In [1]:
import sys
sys.path.insert(0, '..')
sys.path.insert(1, '../../..')
sys.path.insert(0, '../../src')

In [2]:
from perturbation_logic.dataset_statistics import compute_dataset_statistics

csv_path="../../../data/BPI Challenge 2017.csv"

properties = {
    'case_name' : 'case:concept:name',
    'concept_name' : 'concept:name',
    'timestamp_name' : 'time:timestamp',
    'time_since_case_start_column' : 'case_elapsed_time',
    'time_since_last_event_column' : 'event_elapsed_time',
    'day_in_week_column' : 'day_in_week',
    'seconds_in_day_column' : 'seconds_in_day',
    'min_suffix_size' : 5,
    'train_validation_size' : 0.15,
    'test_validation_size' : 0.2,
    'window_size' : 'auto',
    'categorical_columns' : ['concept:name', 'Action', 'org:resource', 'EventOrigin', 'lifecycle:transition', 'case:LoanGoal', 'case:ApplicationType', 'Accepted', 'Selected', ],
    'continuous_columns' : ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_day', 'case:RequestedAmount', 'FirstWithdrawalAmount', 'NumberOfTerms', 'MonthlyCost', 'CreditScore'],
    'continuous_positive_columns' : []
}

stats = compute_dataset_statistics(csv_path, properties)

In [3]:
from datetime import timedelta

duration_str = str(timedelta(seconds=int(stats['avg_case_duration_seconds'])))

print("Dataset Statistics")
print("-" * 40)
print(f"Number of Events:              {stats['n_events']}")
print(f"Number of Cases:               {stats['n_cases']}")
print(f"Number of Activities:          {stats['n_activities']}")
print(f"Average Case Length:           {stats['avg_case_length']:.2f}")
print(f"Average Case Duration:         {duration_str}")
print(f"Number of Static Categorical:  {stats['n_static_categorical']}")
print(f"Number of Static Numerical:    {stats['n_static_numerical']}")
print(f"Number of Dynamic Categorical: {stats['n_dynamic_categorical']}")
print(f"  {stats['dynamic_categorical_names']}")
print(f"Number of Dynamic Numerical:   {stats['n_dynamic_numerical']}")
print(f"  {stats['dynamic_numerical_names']}")

Dataset Statistics
----------------------------------------
Number of Events:              1359812
Number of Cases:               31509
Average Case Length:           43.16
Average Case Duration:         21 days, 21:35:25
Number of Static Categorical:  3
Number of Static Numerical:    1
Number of Dynamic Categorical: 6
  ['concept:name', 'Action', 'org:resource', 'EventOrigin', 'lifecycle:transition', 'Selected']
Number of Dynamic Numerical:   8
  ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_day', 'FirstWithdrawalAmount', 'NumberOfTerms', 'MonthlyCost', 'CreditScore']


In [4]:
from event_log_loader_service.event_log_loader import CSV2EventLog

_event_log = CSV2EventLog(csv_path, **properties)
_df = _event_log.df
_case_col = properties['case_name']
_total_cases = stats['n_cases']

_all_cols = (
    list(properties.get('categorical_columns', []))
    + list(properties.get('continuous_columns', []))
    + list(properties.get('continuous_positive_columns', []))
)
_total_attrs = sum(1 for c in _all_cols if c in _df.columns)

_dynamic_details = stats['dynamic_details']
_static_details = stats['static_details']

_example_case = _df[_case_col].iloc[0]

print("=" * 80)
print("DYNAMIC ATTRIBUTES ANALYSIS")
print("=" * 80)
print(f"\nTotal cases analyzed: {_total_cases}")
print(f"Total attributes checked: {_total_attrs}")
print(f"Threshold for dynamic: > 5.0% of cases with change")
print(f"\nFound {len(_dynamic_details)} dynamic attributes:\n")

for col, num_varies, pct, is_cat in _dynamic_details:
    feat_type = "Categorical" if is_cat else "Continuous"
    print(f"{col} ({feat_type})")
    print(f"  Varies in {num_varies}/{_total_cases} cases ({pct:.1f}%)")
    case_vals = _df.loc[_df[_case_col] == _example_case, col].dropna().tolist()
    vals_str = str(case_vals)
    if len(vals_str) > 80:
        vals_str = vals_str[:77] + "..."
    print(f"  Example from case {_example_case}:")
    print(f"    Values: {vals_str}")
    print()

print("=" * 80)
print("\nStatic attributes (do not change within cases):")
for col, is_cat in _static_details:
    feat_type = "Categorical" if is_cat else "Continuous"
    print(f"  - {col} ({feat_type})")
print("=" * 80)


DYNAMIC ATTRIBUTES ANALYSIS

Total cases analyzed: 31509
Total attributes checked: 18
Threshold for dynamic: > 5.0% of cases with change

Found 14 dynamic attributes:

concept:name (Categorical)
  Varies in 31509/31509 cases (100.0%)
  Example from case Application_1000086665:
    Values: ['A_Create Application', 'A_Submitted', 'W_Handle leads', 'W_Handle leads', '...

Action (Categorical)
  Varies in 31509/31509 cases (100.0%)
  Example from case Application_1000086665:
    Values: ['Created', 'statechange', 'Created', 'Deleted', 'Created', 'statechange', 'O...

org:resource (Categorical)
  Varies in 31438/31509 cases (99.8%)
  Example from case Application_1000086665:
    Values: ['User_1', 'User_1', 'User_1', 'User_1', 'User_1', 'User_1', 'User_14', 'User...

EventOrigin (Categorical)
  Varies in 31509/31509 cases (100.0%)
  Example from case Application_1000086665:
    Values: ['Application', 'Application', 'Workflow', 'Workflow', 'Workflow', 'Applicati...

lifecycle:transition (Ca

In [5]:
from pm4py.statistics.variants.pandas.get import get_variants_count
from pm4py.objects.log.util.pandas_numpy_variants import Parameters

_df_no_eos = _df[_df[properties['concept_name']] != 'EOS'].copy()

_variants = get_variants_count(
    _df_no_eos,
    parameters={
        Parameters.CASE_ID_KEY: properties['case_name'],
        Parameters.ACTIVITY_KEY: properties['concept_name'],
        Parameters.TIMESTAMP_KEY: properties['timestamp_name'],
    },
)

print(f"Number of Variants (PM4Py):    {len(_variants)}")


Number of Variants (PM4Py):    15930
